In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc gem-suite matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# การเปลี่ยนเฟส Nishimori

import TutorialFeedback from '@site/src/components/TutorialFeedback';






*ประมาณการใช้งาน: 3 นาทีบนโปรเซสเซอร์ Heron r2 (หมายเหตุ: นี่เป็นการประมาณการเท่านั้น เวลาที่ใช้จริงอาจแตกต่างกัน)*
## ผลลัพธ์การเรียนรู้
หลังจากผ่านบทช่วยสอนนี้แล้ว ผู้ใช้ควรคาดหวังผลลัพธ์ดังต่อไปนี้:
- เข้าใจการเปลี่ยนเฟส Nishimori และวิธีที่มันแสดงออกมาในรูปของการปรากฏของการพันกันระยะยาวในแบบจำลอง random-bond Ising
- นำโปรโตคอล *การสร้างการพันกันด้วยการวัด* (GEM) ไปใช้บนฮาร์ดแวร์ควอนตัมโดยใช้การวัดกลางวงจรและ Circuit ความลึกคงที่
- ระบุลักษณะการเปลี่ยนผ่านโดยการดึงข้อมูลสหสัมพันธ์สองจุดและความแปรปรวนที่ปรับให้เป็นมาตรฐานของค่าแม่เหล็กจากข้อมูลการทดลอง

## ข้อกำหนดเบื้องต้น
แนะนำให้มีความคุ้นเคยกับหัวข้อต่อไปนี้ก่อนผ่านบทช่วยสอนนี้:
- คู่มือ [Measure qubits](/guides/measure-qubits) โดยเฉพาะส่วนเกี่ยวกับการวัดกลางวงจรที่โปรโตคอล GEM อาศัย
- [การจำลองแบบแม่นยำและมีสัญญาณรบกวนด้วย Qiskit Aer primitives](/guides/simulate-with-qiskit-aer) ซึ่งเป็นวิธีที่ส่วนขนาดเล็กถูกดำเนินการ
- [การพันกันระยะยาวด้วย dynamic circuits](/tutorials/long-range-entanglement) บทช่วยสอนคู่ที่ใช้กรอบการสร้างการพันกันด้วยการวัดแบบเดียวกัน
- [Heavy-hex lattice](https://www.ibm.com/quantum/blog/heavy-hex-lattice) โทโพโลยีฮาร์ดแวร์ IBM&reg; ที่ plaquette lattice ถูกสร้างขึ้นมาจาก
## พื้นหลัง
บทช่วยสอนนี้แสดงวิธีการสร้างการเปลี่ยนเฟส Nishimori บนโปรเซสเซอร์ควอนตัม การทดลองนี้ถูกอธิบายไว้ครั้งแรกใน [*Realizing the Nishimori transition across the error threshold for constant-depth quantum circuits*](https://arxiv.org/abs/2309.02863)

การเปลี่ยนเฟส Nishimori หมายถึงการเปลี่ยนผ่านระหว่างเฟสที่มีลำดับระยะสั้นและระยะยาวในแบบจำลอง random-bond Ising บนคอมพิวเตอร์ควอนตัม เฟสที่มีลำดับระยะยาวแสดงออกมาในรูปของสถานะที่ Qubit ต่างๆ มีการพันกันทั่วทั้งอุปกรณ์ สถานะที่พันกันสูงนี้ถูกเตรียมโดยใช้โปรโตคอล *การสร้างการพันกันด้วยการวัด* (GEM) โดยใช้การวัดกลางวงจร โปรโตคอล GEM สามารถพัน Qubit ทั่วทั้งอุปกรณ์ได้โดยใช้ Circuit ที่มีความลึกคงที่เท่านั้น บทช่วยสอนนี้ใช้การนำโปรโตคอล GEM ไปใช้จากซอฟต์แวร์แพ็กเกจ [GEM Suite](https://github.com/qiskit-community/gem-suite)
## ข้อกำหนด
ก่อนเริ่มบทช่วยสอนนี้ ให้แน่ใจว่าคุณมีสิ่งต่อไปนี้ติดตั้งอยู่:

- Qiskit SDK v1.0 หรือใหม่กว่า พร้อมการสนับสนุน [การแสดงผล](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 หรือใหม่กว่า (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.14 หรือใหม่กว่า (`pip install qiskit-aer`)
- GEM Suite (`pip install gem-suite`)
## การตั้งค่า

In [1]:
import matplotlib.pyplot as plt
import warnings

from collections import defaultdict

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator

from qiskit.transpiler import generate_preset_pass_manager

from gem_suite import PlaquetteLattice
from gem_suite.experiments import GemExperiment

## ตัวอย่างบน Simulator ขนาดเล็ก

ในส่วนนี้ เวิร์กโฟลว์ทั้งหมดจะถูกทดสอบบน `AerSimulator` แบบไม่มีสัญญาณรบกวน plaquette lattice ถูกจำกัดให้เหลือเพียง plaquette เดียว (12 Qubit) เพื่อให้การจำลองมีขนาดเล็กและเร็ว ในขณะที่ยังครอบคลุมทุกส่วนของโปรโตคอล GEM ได้แก่ การวัดกลางวงจร การสแกนมุม $R_{ZZ}$ การถอดรหัส และการวิเคราะห์ความแปรปรวนที่ปรับให้เป็นมาตรฐาน เวิร์กโฟลว์เดียวกันนี้จะถูกขยายไปยัง plaquette หลายอันและ plaquette เต็มบนฮาร์ดแวร์จริงในภายหลัง

### ขั้นตอนที่ 1: แปลงอินพุตแบบคลาสสิกเป็นปัญหาควอนตัม

โปรโตคอล GEM ทำงานบนโปรเซสเซอร์ควอนตัมที่มีการเชื่อมต่อ Qubit ที่อธิบายด้วย lattice โปรเซสเซอร์ IBM Quantum&reg; ในปัจจุบันใช้ [heavy-hex lattice](https://www.ibm.com/quantum/blog/heavy-hex-lattice) Qubit ของโปรเซสเซอร์ถูกจัดกลุ่มเป็น *plaquette* ตามเซลล์หน่วยของ lattice ที่มันอยู่ เนื่องจาก Qubit หนึ่งอาจอยู่ในเซลล์หน่วยมากกว่าหนึ่งเซลล์ plaquette จึงไม่แยกออกจากกัน บน heavy-hex lattice แต่ละ plaquette มี 12 Qubit plaquette เองก็ก่อตัวเป็น lattice เช่นกัน โดย plaquette สองอันจะเชื่อมต่อกันหากมี Qubit ร่วมกัน บน heavy-hex lattice plaquette ที่อยู่ข้างกันจะมี Qubit ร่วมกัน 3 ตัว

ในซอฟต์แวร์แพ็กเกจ GEM Suite คลาสพื้นฐานสำหรับการนำโปรโตคอล GEM ไปใช้คือ `PlaquetteLattice` ซึ่งแทน lattice ของ plaquette (ซึ่งแตกต่างจาก heavy-hex lattice) `PlaquetteLattice` สามารถเริ่มต้นได้จาก qubit coupling map ปัจจุบันรองรับเฉพาะ coupling map แบบ heavy-hex เท่านั้น

โค้ดต่อไปนี้เริ่มต้น plaquette lattice จาก coupling map ของหน่วยประมวลผลควอนตัม (QPU) plaquette lattice ไม่ได้ครอบคลุมฮาร์ดแวร์ทั้งหมดเสมอไป ตัวอย่างเช่น `ibm_torino` มี Qubit ทั้งหมด 133 ตัว แต่ plaquette lattice ที่ใหญ่ที่สุดที่พอดีกับอุปกรณ์นี้ใช้เพียง 125 ตัวเท่านั้น ประกอบด้วย plaquette ทั้งหมด 18 อัน ส่วน `ibm_pittsburgh` (156 Qubit) ก็ใส่ได้ 144 Qubit ใน 21 plaquette เช่นกัน รูปแบบเดียวกันนี้ใช้ได้กับ QPU แบบ heavy-hex อื่นๆ ที่มีจำนวน Qubit ต่างกัน

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="<YOUR_API_KEY>", overwrite=True,
# set_as_default=True)
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
aer_backend = AerSimulator.from_backend(backend)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

print(f"Number of qubits in backend: {backend.num_qubits}")
print(
    f"Number of qubits in plaquette lattice: {len(list(plaquette_lattice.qubits()))}"
)
print(f"Number of plaquettes: {len(list(plaquette_lattice.plaquettes()))}")

คุณสามารถแสดงผล plaquette lattice ได้โดยสร้างแผนภาพของการแทนกราฟ ในแผนภาพ plaquette จะแสดงด้วยรูปหกเหลี่ยมที่มีป้ายกำกับ และ plaquette สองอันจะเชื่อมต่อกันด้วยขอบหากมี Qubit ร่วมกัน

In [3]:
plaquette_lattice.draw_plaquettes()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif)

คุณสามารถดึงข้อมูลเกี่ยวกับ plaquette แต่ละอัน เช่น Qubit ที่มีอยู่ โดยใช้เมธอด `plaquettes`

In [4]:
# Get a list of the plaquettes
plaquettes = list(plaquette_lattice.plaquettes())
# Display information about plaquette 0
plaquettes[0]

PyPlaquette(index=0, qubits=[3, 4, 5, 6, 7, 16, 17, 23, 24, 25, 26, 27], neighbors=[4, 3, 1])

You can also produce a diagram of the underlying qubits that form the plaquette lattice.

In [5]:
plaquette_lattice.draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif" alt="Output of the previous code cell" />

คุณยังสามารถสร้างแผนภาพของ Qubit พื้นฐานที่ก่อตัวเป็น plaquette lattice ได้

In [6]:
# Filter the plaquette lattice down to a single plaquette (12 qubits)
# so the AerSimulator run stays fast. The full lattice is used later
# in the large-scale hardware example.
gem_exp = GemExperiment(plaquette_lattice.filter([9]), backend=aer_backend)

# visualize the plaquette lattice after filtering
plaquette_lattice.filter([9]).draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif)

นอกจากป้ายกำกับ Qubit และขอบที่แสดงว่า Qubit ตัวใดเชื่อมต่อกัน แผนภาพยังมีข้อมูลเพิ่มเติมสามส่วนที่เกี่ยวข้องกับโปรโตคอล GEM:
- Qubit แต่ละตัวจะมีเฉดสี (สีเทา) หรือไม่มีเฉดสี Qubit ที่มีเฉดสีคือ Qubit "ไซต์" ที่แทนไซต์ของแบบจำลอง Ising และ Qubit ที่ไม่มีเฉดสีคือ Qubit "พันธะ" ที่ใช้เป็นตัวกลางสำหรับปฏิสัมพันธ์ระหว่าง Qubit ไซต์
- Qubit ไซต์แต่ละตัวจะถูกติดป้ายว่า (A) หรือ (B) ซึ่งบ่งบอกถึงหนึ่งในสองบทบาทที่ Qubit ไซต์สามารถมีได้ในโปรโตคอล GEM (บทบาทจะอธิบายในภายหลัง)
- แต่ละขอบจะถูกระบายสีโดยใช้หนึ่งในหกสี ทำให้ขอบถูกแบ่งออกเป็นหกกลุ่ม การแบ่งพาร์ทิชันนี้กำหนดวิธีที่ Gate สองคิวบิตสามารถทำแบบขนานได้ รวมถึงรูปแบบการจัดกำหนดการต่างๆ ที่มีแนวโน้มจะเกิดข้อผิดพลาดในปริมาณต่างกันบนโปรเซสเซอร์ควอนตัมที่มีสัญญาณรบกวน เนื่องจากขอบในกลุ่มเดียวกันไม่มีจุดร่วม จึงสามารถใช้ชั้นของ Gate สองคิวบิตกับขอบเหล่านั้นพร้อมกันได้ จริงๆ แล้ว สามารถแบ่งสีหกสีออกเป็นสามกลุ่ม กลุ่มละสองสี ได้ โดยที่สหภาพของแต่ละกลุ่มสองสียังคงไม่มีจุดร่วม ดังนั้น จำเป็นต้องใช้เพียงสามชั้นของ Gate สองคิวบิตเพื่อเปิดใช้งานทุกขอบ มี 12 วิธีในการแบ่งพาร์ทิชันหกสีดังกล่าว และแต่ละพาร์ทิชันให้กำหนดการ Gate 3 ชั้นที่แตกต่างกัน

เมื่อสร้าง plaquette lattice แล้ว ขั้นตอนถัดไปคือการเริ่มต้นออบเจ็กต์ `GemExperiment` โดยส่งทั้ง plaquette lattice และ Backend ที่ต้องการรันการทดลองนั้น คลาส `GemExperiment` จัดการการนำโปรโตคอล GEM ไปใช้จริง รวมถึงการสร้าง Circuit การส่งงาน และการวิเคราะห์ข้อมูล โค้ดต่อไปนี้เริ่มต้นคลาสการทดลองขณะจำกัด plaquette lattice ให้เหลือเพียง plaquette เดียว (12 Qubit) เพื่อให้การจำลองมีขนาดเล็กและเร็ว plaquette lattice เต็มจะถูกใช้ในภายหลังเมื่อขยายไปยังฮาร์ดแวร์จริง

In [7]:
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

Total number of circuits: 252


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif)

Circuit ของโปรโตคอล GEM ถูกสร้างขึ้นโดยใช้ขั้นตอนต่อไปนี้:
1. เตรียมสถานะ all-$|+\rangle$ โดยใช้ Hadamard Gate กับ Qubit ทุกตัว
2. ใช้ $R_{ZZ}$ Gate ระหว่าง Qubit ที่เชื่อมต่อกันทุกคู่ ซึ่งทำได้โดยใช้สามชั้นของ Gate แต่ละ $R_{ZZ}$ Gate ทำงานบน Qubit ไซต์และ Qubit พันธะ หาก Qubit ไซต์ถูกติดป้าย (B) มุมจะถูกกำหนดไว้ที่ $\frac{\pi}{2}$ หาก Qubit ไซต์ถูกติดป้าย (A) มุมจะได้รับอนุญาตให้เปลี่ยนแปลงได้ ทำให้ Circuit แตกต่างกัน โดยค่าเริ่มต้น ช่วงของมุมถูกตั้งค่าให้เป็น 21 จุดที่เว้นระยะเท่ากันระหว่าง $0$ ถึง $\frac{\pi}{2}$ รวมจุดสุดท้าย
3. วัด Qubit พันธะแต่ละตัวในฐาน Pauli $X$ เนื่องจาก Qubit ถูกวัดในฐาน Pauli $Z$ จึงสามารถทำได้โดยใช้ Hadamard Gate ก่อนวัด Qubit

สังเกตว่าเอกสารอ้างอิงในบทนำของบทช่วยสอนนี้ใช้แบบแผนที่ต่างกันสำหรับมุม $R_{ZZ}$ ซึ่งแตกต่างจากแบบแผนที่ใช้ในบทช่วยสอนนี้ด้วยปัจจัย 2

ในขั้นตอนที่ 3 มีเพียง Qubit พันธะเท่านั้นที่ถูกวัด เพื่อเข้าใจว่า Qubit ไซต์อยู่ในสถานะอะไร เป็นเรื่องน่าพิจารณาถึงกรณีที่มุม $R_{ZZ}$ ที่ใช้กับ Qubit ไซต์ (A) ในขั้นตอนที่ 2 เท่ากับ $\frac{\pi}{2}$ ในกรณีนี้ Qubit ไซต์จะอยู่ในสถานะที่พันกันสูงคล้ายกับสถานะ GHZ

$$
\lvert \text{GHZ} \rangle = \lvert 00 \cdots 00 \rangle + \lvert 11 \cdots 11 \rangle.
$$

เนื่องจากความสุ่มในผลลัพธ์การวัด สถานะจริงของ Qubit ไซต์อาจเป็นสถานะที่ต่างกันที่มีลำดับระยะยาว ตัวอย่างเช่น $\lvert 00110 \rangle + \lvert 11001 \rangle$ อย่างไรก็ตาม สถานะ GHZ สามารถกู้คืนได้โดยการใช้การดำเนินการถอดรหัสตามผลการวัด เมื่อมุม $R_{ZZ}$ ถูกปรับลดจาก $\frac{\pi}{2}$ ลำดับระยะยาวยังคงกู้คืนได้จนถึงมุมวิกฤต ซึ่งในกรณีที่ไม่มีสัญญาณรบกวนอยู่ที่ประมาณ $0.3 \pi$ ต่ำกว่ามุมนี้ สถานะที่ได้จะไม่แสดงการพันกันระยะยาวอีกต่อไป การเปลี่ยนผ่านระหว่างการมีและไม่มีลำดับระยะยาวนี้คือการเปลี่ยนเฟส Nishimori

ในคำอธิบายข้างต้น Qubit ไซต์ถูกปล่อยไว้โดยไม่ได้วัด และการดำเนินการถอดรหัสสามารถทำได้โดยการใช้ Gate ควอนตัม ในการทดลองตามที่นำไปใช้ใน GEM suite Qubit ไซต์ถูกวัดจริงๆ และการดำเนินการถอดรหัสจะถูกนำไปใช้ในขั้นตอนหลังประมวลผลแบบคลาสสิก

ในคำอธิบายข้างต้น การดำเนินการถอดรหัสสามารถทำได้โดยการใช้ Gate ควอนตัมกับ Qubit ไซต์เพื่อกู้คืนสถานะควอนตัม อย่างไรก็ตาม หากเป้าหมายคือการวัดสถานะทันที (เช่น เพื่อวัตถุประสงค์ในการระบุลักษณะ) คุณสามารถวัด Qubit ไซต์พร้อมกับ Qubit พันธะและนำการดำเนินการถอดรหัสไปใช้ในขั้นตอนหลังประมวลผลแบบคลาสสิกได้

นอกจากขึ้นอยู่กับมุม $R_{ZZ}$ ในขั้นตอนที่ 2 ซึ่งโดยค่าเริ่มต้นสแกนผ่าน 21 ค่า Circuit ของโปรโตคอล GEM ยังขึ้นอยู่กับรูปแบบการจัดกำหนดการที่ใช้ในการนำสามชั้นของ $R_{ZZ}$ Gate ไปใช้ด้วย ดังที่กล่าวไว้ก่อนหน้า มี 12 รูปแบบการจัดกำหนดการดังกล่าว ดังนั้น จำนวน Circuit ทั้งหมดในการทดลองคือ $21 \times 12 = 252$

Circuit ของการทดลองสามารถสร้างได้โดยใช้เมธอด `circuits` ของคลาส `GemExperiment`

In [8]:
# Restrict experiment to the first scheduling pattern
gem_exp.set_experiment_options(schedule_idx=0)

# There are less circuits now
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

# Print the RZZ angles swept over
print(f"RZZ angles:\n{gem_exp.parameters()}")

Total number of circuits: 21
RZZ angles:
[0.         0.07853982 0.15707963 0.23561945 0.31415927 0.39269908
 0.4712389  0.54977871 0.62831853 0.70685835 0.78539816 0.86393798
 0.9424778  1.02101761 1.09955743 1.17809725 1.25663706 1.33517688
 1.41371669 1.49225651 1.57079633]


The following code cell draws a diagram of the circuit at index 5. To reduce the size of the diagram, the measurement gates at the end of the circuit are removed.

In [9]:
# Get the circuit at index 5
circuit = circuits[5]
# Remove the final measurements to ease visualization
circuit.remove_final_measurements()
# Draw the circuit
circuit.draw("mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif" alt="Output of the previous code cell" />

สำหรับจุดประสงค์ของบทช่วยสอนนี้ การพิจารณารูปแบบการจัดกำหนดการเพียงรูปแบบเดียวก็เพียงพอ โค้ดต่อไปนี้จำกัดการทดลองให้เหลือรูปแบบการจัดกำหนดการแรก ส่งผลให้การทดลองมีเพียง 21 Circuit หนึ่งสำหรับมุม $R_{ZZ}$ แต่ละค่าที่สแกนผ่าน

In [10]:
# Demonstrate setting transpile options
gem_exp.set_transpile_options(
    optimization_level=1  # This is the default optimization level
)
pass_manager = generate_preset_pass_manager(
    backend=aer_backend,
    initial_layout=list(gem_exp.physical_qubits),
    **dict(gem_exp.transpile_options),
)
transpiled = pass_manager.run(circuit)
transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

To execute the GEM protocol circuits on the hardware, call the `run` method of the `GemExperiment` object. You can specify the number of shots you want to sample from each circuit. The `run` method returns an [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) object which you should save to a variable. Note that the `run` method only submits jobs without waiting for them to finish, so it is a non-blocking call.

In [11]:
exp_data = gem_exp.run(shots=10_000)

โค้ดต่อไปนี้วาดแผนภาพของ Circuit ที่อินเด็กซ์ 5 เพื่อลดขนาดของแผนภาพ เกตการวัดที่ท้าย Circuit จะถูกลบออก

In [12]:
# The noiseless AerSimulator produces zero-variance UFloat objects in the
# analysis, which triggers a harmless warning from the `uncertainties`
# library. Suppress it so the output stays clean.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message="Using UFloat objects with std_dev==0"
    )
    exp_data.block_for_results()
exp_data

ExperimentData(GemExperiment, 90bf2a90-f729-4c4e-a6da-664aecb11039, job_ids=['04a7c405-47fd-46ca-aa4b-aaf7e339cfbe'], metadata=<5 items>, figure_names=['two_point_correlation.svg', 'normalized_variance.svg', 'plaquette_ops.svg', 'bond_ops.svg'])

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif)

### ขั้นตอนที่ 2: ปรับปรุงปัญหาสำหรับการประมวลผลบนฮาร์ดแวร์ควอนตัม
การ Transpile Circuit ควอนตัมสำหรับการรันบนฮาร์ดแวร์มักเกี่ยวข้องกับ [หลายขั้นตอน](/guides/transpiler-stages) โดยทั่วไป ขั้นตอนที่มีโอเวอร์เฮดการคำนวณมากที่สุดคือการเลือกเลย์เอาต์ Qubit การกำหนดเส้นทาง Gate สองคิวบิตให้สอดคล้องกับการเชื่อมต่อ Qubit ของฮาร์ดแวร์ และการปรับแต่ง Circuit เพื่อลดจำนวน Gate และความลึก ในโปรโตคอล GEM ขั้นตอนเลย์เอาต์และการกำหนดเส้นทางไม่จำเป็น เนื่องจากการเชื่อมต่อฮาร์ดแวร์ถูกรวมไว้ในการออกแบบโปรโตคอลแล้ว Circuit มีเลย์เอาต์ Qubit อยู่แล้ว และ Gate สองคิวบิตถูกแมปกับการเชื่อมต่อ native แล้ว นอกจากนี้ เพื่อรักษาโครงสร้างของ Circuit ขณะที่มุม $R_{ZZ}$ เปลี่ยนแปลง ควรทำการปรับแต่ง Circuit เพียงขั้นพื้นฐานเท่านั้น

คลาส `GemExperiment` ทำการ Transpile Circuit อย่างโปร่งใสเมื่อรันการทดลอง ขั้นตอนเลย์เอาต์และการกำหนดเส้นทางถูกแทนที่โดยค่าเริ่มต้นให้ไม่ทำอะไร และการปรับแต่ง Circuit จะทำในระดับที่ปรับแต่งเฉพาะ Gate คิวบิตเดี่ยวเท่านั้น อย่างไรก็ตาม คุณสามารถแทนที่หรือส่งตัวเลือกเพิ่มเติมโดยใช้เมธอด `set_transpile_options` เพื่อแสดงให้เห็นด้านภาพ โค้ดต่อไปนี้ทำการ Transpile Circuit ที่แสดงก่อนหน้าด้วยตนเองและวาด Circuit ที่ถูก Transpile แล้ว

In [13]:
def magnetization_distribution(
    counts_dict: dict[str, int],
) -> dict[str, float]:
    """Compute magnetization distribution from counts dictionary."""
    # Construct dictionary from magnetization to count
    mag_dist = defaultdict(float)
    for bitstring, count in counts_dict.items():
        mag = bitstring.count("0") - bitstring.count("1")
        mag_dist[mag] += count
    # Normalize
    shots = sum(counts_dict.values())
    for mag in mag_dist:
        mag_dist[mag] /= shots
    return mag_dist


# Get counts dictionaries with and without decoding
data = exp_data.data()
# Get the last data point, which is at the angle for the GHZ state
raw_counts = data[-1]["counts"]
# Without decoding
site_indices = [
    i for i, q in enumerate(gem_exp.plaquettes.qubits()) if q.role == "Site"
]
site_raw_counts = defaultdict(int)
for key, val in raw_counts.items():
    site_str = "".join(key[-1 - i] for i in site_indices)
    site_raw_counts[site_str] += val
# With decoding
_, site_decoded_counts = gem_exp.plaquettes.decode_outcomes(
    raw_counts, return_counts=True
)

# Compute magnetization distribution
raw_magnetization = magnetization_distribution(site_raw_counts)
decoded_magnetization = magnetization_distribution(site_decoded_counts)

# Plot
plt.bar(*zip(*raw_magnetization.items()), label="raw")
plt.bar(*zip(*decoded_magnetization.items()), label="decoded", width=0.3)
plt.legend()
plt.xlabel("Magnetization")
plt.ylabel("Frequency")
plt.title("Magnetization distribution with and without decoding")

Text(0.5, 1.0, 'Magnetization distribution with and without decoding')

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/8ead3582-16df-4616-836c-bdce867ad6b8-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif)

### ขั้นตอนที่ 3: ประมวลผลโดยใช้ Qiskit primitives
เพื่อรัน Circuit ของโปรโตคอล GEM บนฮาร์ดแวร์ ให้เรียกเมธอด `run` ของออบเจ็กต์ `GemExperiment` คุณสามารถระบุจำนวน shot ที่ต้องการสุ่มตัวอย่างจากแต่ละ Circuit เมธอด `run` คืนค่าออบเจ็กต์ [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) ซึ่งควรบันทึกไว้ในตัวแปร สังเกตว่าเมธอด `run` เพียงแค่ส่งงานโดยไม่รอให้เสร็จ ดังนั้นจึงเป็นการเรียกแบบ non-blocking

In [14]:
exp_data.figure("two_point_correlation")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/4ecb25c8-e572-49af-a879-9943039db131-0.avif" alt="Output of the previous code cell" />

เพื่อรอผลลัพธ์ ให้เรียกเมธอด `block_for_results` ของออบเจ็กต์ `ExperimentData` การเรียกนี้จะทำให้ตัวแปลโปรแกรมหยุดรอจนกว่างานจะเสร็จ

In [15]:
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/2b351d68-3924-445a-94ef-047b16214e8a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

Having validated the protocol on a simulator, you can now scale up the experiment and run it on the real quantum hardware backend selected in the [Setup](#setup) section. This example uses two larger problem sizes:

- **Six plaquettes (~49 qubits)**: a mid-size run that already shows the rightward shift of the critical point under hardware noise.
- **The full plaquette lattice**: every plaquette the device's heavy-hex topology supports (for example, 18 plaquettes / 125 qubits on `ibm_torino` or 21 plaquettes / 144 qubits on `ibm_pittsburgh`), entangling qubits across the entire device with constant-depth circuits.

The single code cell below is self-contained: it builds the plaquette lattice from the backend's coupling map and runs both experiments, so this section can be executed after the [Setup](#setup) cells without first running the small-scale section.

In [ ]:
# -------------------------Step 1-------------------------
# Initialize the runtime service, pick a real quantum hardware backend,
# and build the plaquette lattice from its coupling map. This is repeated
# from the small-scale example so this cell can run standalone after the
# Setup section. The full plaquette lattice is the "large-scale" target;
# a six-plaquette subset (range(3, 9)) is also used to show an intermediate
# scaling step.
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

# Build a GemExperiment for the full plaquette lattice and one for the
# six-plaquette subset, each restricted to a single scheduling pattern so
# the experiment has one circuit per RZZ angle (21 circuits total).
gem_exp_full = GemExperiment(plaquette_lattice, backend=backend)
gem_exp_full.set_experiment_options(schedule_idx=0)
gem_exp_6 = GemExperiment(
    plaquette_lattice.filter(range(3, 9)), backend=backend
)
gem_exp_6.set_experiment_options(schedule_idx=0)

circuits = gem_exp_full.circuits()
print(f"Total number of circuits (full lattice): {len(circuits)}")

# -------------------------Step 2-------------------------
# GemExperiment transpiles internally for the target backend: the layout
# and routing stages are overridden because the plaquette lattice already
# matches the hardware connectivity, and optimization is restricted so the
# RZZ angle structure is preserved. The code below manually transpiles one
# circuit from the six-plaquette experiment with the same settings this
# experiment will use, and draws it for inspection. (The full-lattice
# transpiled circuit has too many qubits to visualize cleanly, so the
# six-plaquette circuit is used here as a representative example.)
gem_exp_6.set_transpile_options(optimization_level=1)
circuits_6 = gem_exp_6.circuits()
pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(gem_exp_6.physical_qubits),
    **dict(gem_exp_6.transpile_options),
)
transpiled = pass_manager.run(circuits_6[5])
display(transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5))

# -------------------------Step 3-------------------------
# Run both problem sizes on real hardware:
#   1. Six plaquettes (~49 qubits) — an intermediate scale-up.
#   2. The full plaquette lattice — every plaquette the device supports.
exp_data_6 = gem_exp_6.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_full = gem_exp_full.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_6.block_for_results()
exp_data_full.block_for_results()

# -------------------------Step 4-------------------------
# Plot the normalized variance at each scale. The peak marks the critical
# point of the Nishimori transition; as the system grows, hardware noise
# shifts the peak rightward.
display(exp_data_6.figure("normalized_variance"))
exp_data_full.figure("normalized_variance")

Total number of circuits (full lattice): 21


<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-1.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-2.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-3.avif" alt="Output of the previous code cell" />

### ขั้นตอนที่ 4: หลังประมวลผลและแสดงผลในรูปแบบคลาสสิกที่ต้องการ
ที่มุม $R_{ZZ}$ เท่ากับ $\frac{\pi}{2}$ สถานะที่ถอดรหัสแล้วจะเป็นสถานะ GHZ ในกรณีที่ไม่มีสัญญาณรบกวน ลำดับระยะยาวของสถานะ GHZ สามารถแสดงได้โดยการพล็อตค่าแม่เหล็กของบิตสตริงที่วัดได้ ค่าแม่เหล็ก $M$ ถูกนิยามเป็นผลรวมของตัวดำเนินการ Pauli $Z$ คิวบิตเดี่ยว
$$
M = \sum_{j=1}^N Z_j,
$$
โดยที่ $N$ คือจำนวน Qubit ไซต์ ค่าสำหรับบิตสตริงหนึ่งเท่ากับความแตกต่างระหว่างจำนวนศูนย์และจำนวนหนึ่ง การวัดสถานะ GHZ จะให้สถานะ all-zeros หรือ all-ones ด้วยความน่าจะเป็นเท่ากัน ดังนั้นค่าแม่เหล็กจะเป็น $+N$ ครึ่งหนึ่งของเวลาและ $-N$ อีกครึ่งหนึ่ง เมื่อมีข้อผิดพลาดจากสัญญาณรบกวน ค่าอื่นก็จะปรากฏขึ้นด้วย แต่ถ้าสัญญาณรบกวนไม่มากเกินไป การกระจายก็ยังคงมียอดอยู่ใกล้ $+N$ และ $-N$

สำหรับบิตสตริงดิบก่อนการถอดรหัส การกระจายของค่าแม่เหล็กจะเทียบเท่ากับบิตสตริงแบบสุ่มสม่ำเสมอในกรณีที่ไม่มีสัญญาณรบกวน

โค้ดต่อไปนี้พล็อตค่าแม่เหล็กของบิตสตริงดิบและบิตสตริงที่ถอดรหัสแล้วที่มุม $R_{ZZ}$ เท่ากับ $\frac{\pi}{2}$